# 03 Relation Extraction With Qwen Hybrid

Build graph/relation artifacts in dry-run mode. No Neo4j or Supabase writes happen in Kaggle.

This notebook must export `payloads/<strategy>/` so local import can happen later without rerunning the relation LLM.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

CORPUS_SLUG = 'tuvi-golden-corpus'
SCRIPTS_SLUG = 'tuvi-battu-scripts'

ALL_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child', 'chunk_semantic_embedding_bge_m3']
SOURCE_IDS = ['TVKL', 'TVNL', 'TVHS', 'TVGM']
RUN_TAG = 'part_a'
PARTITION_MODE = 'strategy'
SELECTED_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child']
SELECTED_SOURCES = list(SOURCE_IDS)

CORPUS_DIR = Path(f'/kaggle/input/{CORPUS_SLUG}/benchmark/tuvi_golden_dataset')
SCRIPTS_DIR = Path(f'/kaggle/input/{SCRIPTS_SLUG}')
if not CORPUS_DIR.exists():
    CORPUS_DIR = Path.cwd() / 'benchmark' / 'tuvi_golden_dataset'
if not SCRIPTS_DIR.exists():
    SCRIPTS_DIR = Path.cwd()

ACTIVE_STRATEGIES = [item for item in ALL_STRATEGIES if item in SELECTED_STRATEGIES]
if not ACTIVE_STRATEGIES:
    raise ValueError('No active strategies selected')

OUTPUT_ROOT = Path('/kaggle/working/w3_local_outputs')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
REPORTS = OUTPUT_ROOT / 'reports'
STATE = OUTPUT_ROOT / 'state'
PAYLOADS = OUTPUT_ROOT / 'payloads'
REPORTS.mkdir(parents=True, exist_ok=True)
STATE.mkdir(parents=True, exist_ok=True)
PAYLOADS.mkdir(parents=True, exist_ok=True)

print('CORPUS_DIR =', CORPUS_DIR)
print('SCRIPTS_DIR =', SCRIPTS_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print(json.dumps({'run_tag': RUN_TAG, 'partition_mode': PARTITION_MODE, 'active_strategies': ACTIVE_STRATEGIES, 'selected_sources_recorded_only': SELECTED_SOURCES}, ensure_ascii=False, indent=2))

In [ ]:
for strategy in ACTIVE_STRATEGIES:
    cmd = [
        sys.executable, '-B', str(SCRIPTS_DIR / 'scripts' / 'write_graph_provenance.py'),
        '--chunks-input', str(OUTPUT_ROOT / 'chunks' / strategy),
        '--entities-input', str(OUTPUT_ROOT / 'entities' / strategy / f'{strategy}_entities.jsonl'),
        '--chunking-strategy', strategy,
        '--relation-mode', 'hybrid',
        '--dry-run',
        '--summary-output', str(REPORTS / f'{strategy}_graph_write_summary.json'),
        '--relation-review-output', str(REPORTS / f'{strategy}_relation_review.json'),
        '--payload-output-dir', str(PAYLOADS / strategy),
        '--state-output', str(STATE / f'{strategy}_graph_relation_state.json'),
        '--resume',
        '--llm-backend', 'local',
        '--model', 'Qwen/Qwen2.5-7B-Instruct',
        '--local-llm-quantization', '4bit',
        '--local-llm-max-new-tokens', '1024',
    ]
    print('Running artifact-only graph step for strategy =', strategy)
    subprocess.run(cmd, cwd=SCRIPTS_DIR, check=True)

print('Skipped strategies =', [item for item in ALL_STRATEGIES if item not in ACTIVE_STRATEGIES])

In [ ]:
for strategy in ACTIVE_STRATEGIES:
    summary_path = REPORTS / f'{strategy}_graph_write_summary.json'
    review_path = REPORTS / f'{strategy}_relation_review.json'
    payload_dir = PAYLOADS / strategy
    print(strategy)
    print('  summary =', summary_path.exists(), summary_path)
    print('  review  =', review_path.exists(), review_path)
    print('  payload =', payload_dir.exists(), payload_dir)
print('Artifact-only graph step complete; no Neo4j or Supabase writes were attempted in Kaggle.')